# molprop-ensemble — Notebook Utama

Pipeline ensemble **ECFP+RF · ChemBERTa · D-MPNN** + SMILES-Enumeration TTA (BBBP, BACE, ClinTox).

### Cara pakai — cukup **Run All**
Tidak ada pilihan/centang. Notebook **selalu full run** (3 dataset × 5 seed) dan **otomatis**:
- mengecek apakah ada artefak tersimpan yang **valid** (versi cocok) → dipakai ulang (resume),
- membersihkan artefak **basi** (kalau kode/algoritma berubah) → sekali, otomatis.

Jadi **tidak perlu reset manual**. Kalau sesi terputus, cukup Run All lagi — training lanjut dari checkpoint terakhir.

**Prasyarat panel kanan:** Accelerator = **GPU T4 x2**, Internet = **On**, dan Secret **`GH_TOKEN`** bila repo privat (lihat `KAGGLE.md`).

> ⏱️ Full run memakan waktu lama (bisa berjam-jam). Sesi Kaggle ~9–12 jam; resume otomatis aktif.

## 1. Setup — clone, install, cek environment

In [ ]:
# 1a. Clone repo dari GitHub (pakai GH_TOKEN bila repo privat)
REPO_OWNER, REPO_NAME, REPO_DIR = "belvahector-ship-it", "pharm_", "pharm_"
import os, subprocess
if not os.path.exists(REPO_DIR):
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        pass
    url = (f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git" if token
           else f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git")
    print("clone via GH_TOKEN..." if token else "GH_TOKEN tak ada -> coba clone publik.")
    r = subprocess.run(["git", "clone", url, REPO_DIR], capture_output=True, text=True,
                       timeout=120, stdin=subprocess.DEVNULL)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError("Clone gagal — repo privat tanpa GH_TOKEN valid? Lihat KAGGLE.md.")
else:
    print(f"{REPO_DIR} sudah ada, skip clone.")
%cd {REPO_DIR}
!git pull --quiet && git log --oneline -1

In [ ]:
# 1b. Install dependency (chemprop 2.x; deepchem tidak diperlukan)
!pip install -q rdkit "chemprop>=2.1,<3.0" transformers
print("install selesai")

In [ ]:
# 1c. Cek environment (WAJIB 2 GPU) + versi paket
import torch
print("torch:", torch.__version__, "| GPU:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print("  GPU", i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, "Set Accelerator = GPU T4 x2 di panel kanan, lalu restart."
import rdkit, sklearn, transformers, chemprop
print("rdkit", rdkit.__version__, "| sklearn", sklearn.__version__,
      "| transformers", transformers.__version__, "| chemprop", chemprop.__version__)

## 2. Data & scaffold split
`01_prepare_data` otomatis mengecek versi artefak: valid → dipakai ulang, basi → dibersihkan sekali. Diagnostik memastikan split sah (**overlap SCAFFOLD = 0**).

In [ ]:
# 2a. Fase 1 — scaffold split (auto reuse/refresh berbasis versi)
!python scripts/01_prepare_data.py

In [ ]:
# 2b. Diagnostik mutu split — WAJIB "overlap SCAFFOLD = 0" & "bocor->train = 0.0%"
!python scripts/00_diagnose_split.py

## 3. Jalankan pipeline — FULL (3 dataset × 5 seed)
Satu alur: **training (Fase 4) → TTA (5) → fusion (6) → evaluasi (7)**. ChemBERTa→GPU0, D-MPNN→GPU1, RF→CPU (paralel). Yang sudah ada di-skip/resume otomatis, jadi aman dijalankan ulang bila sesi terputus.

In [ ]:
import subprocess

# Fase 4 — training SEMUA dataset x SEMUA seed, paralel di 2 GPU + CPU.
procs = {m: subprocess.Popen(["python", "scripts/02_train_baselines.py", "--model", m])
         for m in ["chemberta", "dmpnn", "rf"]}
for name, p in procs.items():
    rc = p.wait()
    print(f"[Fase4 {name}] rc={rc}")
    assert rc == 0, f"training {name} gagal (rc={rc}) — cek log di atas"

# Fase 5-7 (semua dataset x seed)
for script in ["04_run_tta.py", "03_run_fusion.py", "05_evaluate.py"]:
    rc = subprocess.run(["python", f"scripts/{script}"]).returncode
    print(f"[{script}] rc={rc}")
    assert rc == 0, f"{script} gagal (rc={rc})"
print("\nPIPELINE FULL SELESAI.")

## 4. Hasil
Tabel final: `roc_auc_mean±std` (5 seed), **bootstrap 95% CI**, **p-value** vs baseline post-hoc, **Cohen's d**. Dengan full run 5 seed, kolom p-value & std kini bermakna.

In [ ]:
import pandas as pd
pd.set_option("display.max_rows", None, "display.width", 200)
pd.read_csv("outputs/results/final_table.csv")

## 5. Simpan hasil
`outputs/` di-gitignore (tak ikut GitHub). Zip lalu unduh dari panel Output, atau **Save Version**.

In [ ]:
!pip freeze > outputs/results/pip_freeze.txt
!cd outputs && zip -rq /kaggle/working/hasil_outputs.zip .
print("tersimpan: /kaggle/working/hasil_outputs.zip  &  outputs/results/{environment,pip_freeze}.txt")